In [1]:
from pyspark.sql import SparkSession

try:
    spark = SparkSession.builder \
        .master("local[*]") \
        .appName("CleanSetup") \
        .getOrCreate()
        
    print(f"✅ Spark Session created successfully! Version: {spark.version}")
    
    data = [("Alice", 1), ("Bob", 2)]
    df = spark.createDataFrame(data, ["Name", "Value"])
    df.show()
    print("✅ Data processed perfectly!")
    
except Exception as e:
    print(f"❌ Error: {e}")

✅ Spark Session created successfully! Version: 4.1.1
❌ Error: An error occurred while calling o48.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, most recent failure: Lost task 0.0 in stage 0.0 (TID 0) (chohjingyi executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:678)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:663)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1034)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(Py

In [2]:
from pyspark.sql import SparkSession

# The safest way to write file paths for Java on Windows
jar_path = "C:\spark-jars\postgresql-42.7.11.jar"

# 1. Boot up a FRESH Spark session and force-feed it the JAR file
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("PostgresETL") \
    .config("spark.jars", jar_path) \
    .config("spark.driver.extraClassPath", jar_path) \
    .getOrCreate()

# 2. Set up the connection details
jdbc_url = "jdbc:postgresql://localhost:5432/postgres" 
connection_properties = {
    "user": "postgres",
    "password": "abc2556670", # <-- Update this!
    "driver": "org.postgresql.Driver"
}

# 3. Test the bridge to the database
try:
    print("Attempting to connect to PostgreSQL...")
    
    test_df = spark.read.jdbc(
        url=jdbc_url,
        table="(SELECT version()) as version_test", 
        properties=connection_properties
    )
    
    test_df.show(truncate=False)
    print("✅ Postgres Connection Successful! Your ETL foundation is complete.")
    
except Exception as e:
    print(f"❌ Connection Failed. Error details:\n{e}")

Attempting to connect to PostgreSQL...


<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
C:\Users\jingy\AppData\Local\Temp\ipykernel_56752\433722531.py:4: SyntaxWarning: invalid escape sequence '\s'
  jar_path = "C:\spark-jars\postgresql-42.7.11.jar"


❌ Connection Failed. Error details:
An error occurred while calling o57.jdbc.
: java.lang.ClassNotFoundException: org.postgresql.Driver
	at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:592)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:525)
	at org.apache.spark.sql.execution.datasources.jdbc.DriverRegistry$.register(DriverRegistry.scala:47)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1(JDBCOptions.scala:112)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1$adapted(JDBCOptions.scala:112)
	at scala.Option.foreach(Option.scala:437)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:112)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:42)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcRelationProvider.createRelation(JdbcRel

In [3]:
import os
import requests
import zipfile
import urllib3

# Suppress the warning Python gives when bypassing SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 1. Setup your folders
data_dir = "C:/Users/jingy/Documents/Spark_ETL_Project/raw_data"
os.makedirs(data_dir, exist_ok=True)

# 2. Define the target
year = 2021
url = f"https://download.inep.gov.br/dados_abertos/microdados_censo_escolar_{year}.zip"
zip_file_path = os.path.join(data_dir, f"censo_{year}.zip")

# 3. Download the file (Notice the verify=False!)
print(f"Downloading {year} data (This might take a few minutes)...")
response = requests.get(url, stream=True, verify=False)

if response.status_code == 200:
    with open(zip_file_path, 'wb') as file:
        for chunk in response.iter_content(chunk_size=8192):
            file.write(chunk)
    print("✅ Download complete!")
    
    # 4. Unzip the file
    print("Extracting files...")
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    print("✅ Extraction complete! The CSV files are ready.")
    
else:
    print(f"❌ Failed to download. The server returned status code: {response.status_code}")

✅ Download complete!
Extracting files...
✅ Extraction complete! The CSV files are ready.


In [4]:
import os
import glob
from pyspark.sql import SparkSession

# 1. Locate the extracted CSV file automatically
data_dir = "C:/Users/jingy/Documents/Spark_ETL_Project/raw_data"

# INEP data usually extracts into a subfolder with a .CSV extension
search_path = os.path.join(data_dir, "**", "*.csv")
search_path_upper = os.path.join(data_dir, "**", "*.CSV")

csv_files = glob.glob(search_path, recursive=True) + glob.glob(search_path_upper, recursive=True)

if not csv_files:
    print("❌ Could not find any CSV files in the raw_data folder.")
else:
    # Grab the largest CSV file (which is the main microdata file)
    target_csv = max(csv_files, key=os.path.getsize)
    print(f"📂 Found target dataset: {target_csv}")
    
    # 2. Boot up Spark (Keeping our Postgres bridge intact for the 'Load' phase later)
    jar_path = "file:///C:/spark-jars/postgresql-42.7.11.jar"
    spark = SparkSession.builder \
        .master("local[*]") \
        .appName("CensusETL_Transform") \
        .config("spark.jars", jar_path) \
        .config("spark.driver.extraClassPath", jar_path) \
        .getOrCreate()
        
    print("⏳ Loading massive CSV into Apache Spark Engine...")
    
    # 3. Read the Data using the specific Brazilian formats
    df = spark.read.csv(
        target_csv,
        header=True,
        sep=";", 
        encoding="iso-8859-1" # <--- Changed from 'latin1'
    )
    
    print("✅ Data successfully loaded into a Spark DataFrame!")
    
    # 4. Let's look at what we are dealing with!
    df.show(5)
    print(f"Total Columns: {len(df.columns)}")
    
    print("Counting total rows (this might take 10-20 seconds)...")
    print(f"Total Rows: {df.count()}")

📂 Found target dataset: C:/Users/jingy/Documents/Spark_ETL_Project/raw_data\microdados_ed_basica_2021\dados\microdados_ed_basica_2021.csv
⏳ Loading massive CSV into Apache Spark Engine...
✅ Data successfully loaded into a Spark DataFrame!
+------------+---------+---------+--------+-----+-----+--------------------+------------+-----------------+--------------+---------------+---------------+-----------+-----------+--------------------+--------------+---------------------------+--------------+---------------------------+--------------------+-----------+---------------+-----------+--------+------+-----------+-------------------------+-----------------+--------------------+---------------------+------------------------------+----------------------------+---------------------------+----------------------+----------------+-------------------------+--------------------------+--------------------------+----------------------------+-----------------------------+---------------------------+-----

In [5]:
from pyspark.sql.functions import monotonically_increasing_id

print("⏳ Slicing data to create dim_location...")

# 1. Select ONLY the location columns, remove duplicates, and add an ID column
dim_location = df.select("NO_REGIAO", "SG_UF", "NO_MUNICIPIO") \
                 .dropDuplicates() \
                 .withColumn("location_id", monotonically_increasing_id())

# Let's see what our new, clean table looks like
dim_location.show(5, truncate=False)
print(f"Total unique locations: {dim_location.count()}")


# 2. LOAD PHASE: Push this new table to PostgreSQL
print("⏳ Loading dim_location into PostgreSQL...")

jdbc_url = "jdbc:postgresql://localhost:5432/postgres" 
connection_properties = {
    "user": "postgres",
    "password": "abc2556670", # <-- CHANGE THIS
    "driver": "org.postgresql.Driver"
}

try:
    dim_location.write.jdbc(
        url=jdbc_url,
        table="public.dim_location",
        mode="overwrite", 
        properties=connection_properties
    )
    print("✅ SUCCESS! dim_location is now stored in your local data warehouse.")
except Exception as e:
    print(f"❌ Failed to write to database. Error: {e}")

⏳ Slicing data to create dim_location...
+---------+-----+------------------+-----------+
|NO_REGIAO|SG_UF|NO_MUNICIPIO      |location_id|
+---------+-----+------------------+-----------+
|Norte    |RO   |Vale do Paraíso   |0          |
|Norte    |RO   |Seringueiras      |1          |
|Norte    |RO   |Urupá             |2          |
|Norte    |AC   |Epitaciolândia    |3          |
|Norte    |AM   |Boa Vista do Ramos|4          |
+---------+-----+------------------+-----------+
only showing top 5 rows
Total unique locations: 5570
⏳ Loading dim_location into PostgreSQL...
❌ Failed to write to database. Error: An error occurred while calling o82.jdbc.
: java.lang.ClassNotFoundException: org.postgresql.Driver
	at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:592)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:525)
	at org.apache.spark.sql.execution.datasources.jdbc.DriverRegistry$.register

In [6]:
print("⏳ Creating dim_school...")

# 1. Build the School Dimension (Using the unique School Code 'CO_ENTIDADE' as the ID)
dim_school = df.select("CO_ENTIDADE", "NO_ENTIDADE", "TP_DEPENDENCIA") \
               .dropDuplicates(["CO_ENTIDADE"]) \
               .withColumnRenamed("CO_ENTIDADE", "school_id")

dim_school.show(5, truncate=False)


print("⏳ Building the central Fact Table...")

# 2. Join the massive dataframe with your dim_location to get the 'location_id'
df_with_location = df.join(dim_location, on=["NO_REGIAO", "SG_UF", "NO_MUNICIPIO"], how="inner")

# 3. Select the keys and the actual metrics we want to analyze
# (e.g., Internet access, filtered water, and total basic education enrollments)
fact_school = df_with_location.select(
    df_with_location["CO_ENTIDADE"].alias("school_id"),
    "location_id",
    "IN_INTERNET",
    "IN_AGUA_FILTRADA",
    "QT_MAT_BAS"
)

fact_school.show(5)


# 4. LOAD PHASE: Push both new tables to PostgreSQL
print("⏳ Loading tables into PostgreSQL...")

jdbc_url = "jdbc:postgresql://localhost:5432/postgres" 
connection_properties = {
    "user": "postgres",
    "password": "abc2556670", # <-- UPDATE THIS
    "driver": "org.postgresql.Driver"
}

try:
    # Write dim_school
    print("Writing dim_school...")
    dim_school.write.jdbc(url=jdbc_url, table="public.dim_school", mode="overwrite", properties=connection_properties)
    
    # Write fact_school
    print("Writing fact_school...")
    fact_school.write.jdbc(url=jdbc_url, table="public.fact_school", mode="overwrite", properties=connection_properties)
    
    print("✅ SUCCESS! Your entire Star Schema is now loaded into Postgres!")
except Exception as e:
    print(f"❌ Failed to write to database. Error: {e}")

⏳ Creating dim_school...
+---------+-----------------------------------------------+--------------+
|school_id|NO_ENTIDADE                                    |TP_DEPENDENCIA|
+---------+-----------------------------------------------+--------------+
|11000198 |COLEGIO SAPIENS - UNIDADE JARDIM DAS MANGUEIRAS|4             |
|11000325 |INSTITUTO EVANGELICO DE EDUCACAO PAUL AENIS    |4             |
|11000546 |EMEF BAIXA VERDE                               |3             |
|11000856 |EEEFM PROFESSOR DANIEL NERI DA SILVA           |2             |
|11001224 |EMEF ERIALDO GOMES DO CARMO                    |3             |
+---------+-----------------------------------------------+--------------+
only showing top 5 rows
⏳ Building the central Fact Table...
+---------+-----------+-----------+----------------+----------+
|school_id|location_id|IN_INTERNET|IN_AGUA_FILTRADA|QT_MAT_BAS|
+---------+-----------+-----------+----------------+----------+
| 11022558|         34|          0|            